# Fundamentos de Ingeniería de Datos
## Modelado: de la capa TRANSFORMED a un *star schema* (capa SERVING)

**Unidad 3 — Temas de hoy:** 3.1 Diseño lógico y físico en SQL · 3.2 Modelos analíticos (*star schema*, hechos y dimensiones).

```
TRANSFORMED                                   SERVING (hoy)
transformed_vg_ventas_ratings.parquet    →    serving.db (DuckDB)
transformed_vg_global_codificado.parquet      df1: dim_*_df1 (6) · fact_ventas_df1
transformed_raw_vg_engagement.parquet         df2: dim_*_df2 (5) · fact_ventas_df2
                                              df3: dim_*_df3 (3) · fact_engagement_df3
```

| Capa | Dónde vive | Qué contiene | *Medallion* |
|---|---|---|---|
| RAW / EXTRACTED | `base_datos.db` (SQLite) | Datos como llegaron | Bronze |
| TRANSFORMED | `transformed_*.parquet` | Datos limpios, **una tabla plana por dataset** | Silver |
| **SERVING** | **`serving.db` (DuckDB)** | **Esquema estrella: hechos + dimensiones** | **Gold** |

| Alias | Parquet | Contenido |
|---|---|---|
| `df1` | `transformed_vg_ventas_ratings` | Ventas por juego y plataforma + ratings |
| `df2` | `transformed_vg_global_codificado` | Ventas codificadas (sin nombre de juego ni año) |
| `df3` | `transformed_raw_vg_engagement` | Engagement de jugadores (sin ventas) |

> Los tres modelos van en el **mismo** `serving.db`. **Ningún nombre de tabla se repite** entre datasets (sufijos `_df1`, `_df2`, `_df3`).

### Configuración del entorno

In [ ]:
# ── Importar utils.py ──────────────────────────────────────────────────────

import sys
import os
sys.path.append(os.path.abspath('../Utilidades'))   # el notebook se ejecuta desde Codigos/

from utils import PROYECTO_BASE, PROCESSED_PATH, MODELADO_PATH, SERVING_DB, ver_serving

print()
print('utils.py importado correctamente')
print()

In [ ]:
# ── Librerías, rutas y conexión ────────────────────────────────────────────

import os
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)


# Origen: la capa TRANSFORMED (Unidad 2)
PARQUET_DF1 = f'{PROCESSED_PATH}/transformed_vg_ventas_ratings.parquet'
PARQUET_DF2 = f'{PROCESSED_PATH}/transformed_vg_global_codificado.parquet'
PARQUET_DF3 = f'{PROCESSED_PATH}/transformed_raw_vg_engagement.parquet'

# Destino: la capa SERVING (MODELADO_PATH y SERVING_DB vienen de utils.py)
os.makedirs(MODELADO_PATH, exist_ok=True)


# Si serving.db no existe, DuckDB lo crea
con = duckdb.connect(SERVING_DB)

print()
print(f'Origen df1 : {PARQUET_DF1}')
print(f'Origen df2 : {PARQUET_DF2}')
print(f'Origen df3 : {PARQUET_DF3}')
print(f'Destino    : {SERVING_DB}')
print()

---
## PASO 1 — Conocer las tablas planas

Creamos una **vista** (`VIEW`) por dataset: guarda solo la consulta, no los datos. Cada vez que usamos `base_df1`, `base_df2` o `base_df3`, DuckDB lee el Parquet.

In [ ]:
# ── Vistas sobre los Parquet ───────────────────────────────────────────────

for vista, parquet in [('base_df1', PARQUET_DF1),
                       ('base_df2', PARQUET_DF2),
                       ('base_df3', PARQUET_DF3)]:
    con.execute(f'''
        CREATE OR REPLACE VIEW {vista} AS
        SELECT * FROM read_parquet('{parquet}')
    ''')

con.sql('SELECT * FROM base_df1 LIMIT 5').df()

In [ ]:
# ── Columnas y tipos (df1) ─────────────────────────────────────────────────

con.sql('DESCRIBE base_df1').df()

In [ ]:
# ── Columnas y tipos (df2) ─────────────────────────────────────────────────

con.sql('DESCRIBE base_df2').df()

In [ ]:
# ── Columnas y tipos (df3) ─────────────────────────────────────────────────

con.sql('DESCRIBE base_df3').df()

In [ ]:
# ── Contenido actual de serving.db (ver_serving() está en utils.py) ────────

ver_serving(SERVING_DB)

---
## PASO 2 — Diseño lógico: ¿qué tablas necesitamos? (3.1)

La tabla plana **repite** el mismo texto en muchas filas. Si una editorial cambia de nombre, habría que corregir cientos de filas (*anomalía de actualización*). **Normalizar** es separar lo que se repite en su propia tabla, para guardarlo una sola vez.

In [ ]:
# ── ¿Cuánto se repite? (df1) ───────────────────────────────────────────────

con.sql('''
    SELECT COUNT(*)                         AS filas,
           COUNT(DISTINCT name)             AS juegos,
           COUNT(DISTINCT platform)         AS plataformas,
           COUNT(DISTINCT publisher)        AS editoriales,
           COUNT(DISTINCT year_of_release)  AS anios
    FROM base_df1
''').df()

In [ ]:
# ── ¿Cuánto se repite? (df2) ───────────────────────────────────────────────

con.sql('''
    SELECT COUNT(*)                         AS filas,
           COUNT(DISTINCT plataforma)       AS plataformas,
           COUNT(DISTINCT genero)           AS generos,
           COUNT(DISTINCT publisher)        AS editoriales,
           COUNT(DISTINCT clasificacion)    AS clasificaciones
    FROM base_df2
''').df()

In [ ]:
# ── ¿Cuánto se repite? (df3) ───────────────────────────────────────────────

con.sql('''
    SELECT COUNT(*)                         AS filas,
           COUNT(DISTINCT title)            AS juegos,
           COUNT(DISTINCT genero_principal) AS generos,
           COUNT(DISTINCT anio)             AS anios
    FROM base_df3
''').df()

Las tres consultas anteriores miran solo las columnas principales. La siguiente cuenta los **valores distintos de todas las columnas** de cada tabla plana: `repeticion` = filas ÷ valores distintos (cuántas veces, en promedio, aparece cada valor).

In [ ]:
# ── ¿Cuánto se repite cada columna? (las tres tablas planas) ───────────────

def repeticion(vista):
    columnas = [c[0] for c in con.sql(f'DESCRIBE {vista}').fetchall()]
    conteos  = ', '.join(f'COUNT(DISTINCT "{c}")' for c in columnas)
    filas, *distintos = con.sql(f'SELECT COUNT(*), {conteos} FROM {vista}').fetchall()[0]
    return pd.DataFrame({'vista': vista, 'columna': columnas, 'valores_distintos': distintos,
                         'repeticion': [round(filas / d, 1) if d else None for d in distintos]})

tabla_repeticion = pd.concat([repeticion(v) for v in ['base_df1', 'base_df2', 'base_df3']],
                             ignore_index=True)

pd.set_option('display.max_rows', 100)
tabla_repeticion

**Decisión de diseño:** miles de filas, pero pocas plataformas, géneros y editoriales. Cada columna que se repite será una **dimensión**; los números serán los **hechos**. **Todas** las columnas de negocio de la tabla plana tienen un destino en el modelo.

Las columnas de texto que describen a la fila y no se repiten lo suficiente para tener su propia tabla (por ejemplo, la región principal o una bandera de calidad) se quedan en el hecho como **atributos degenerados**. De los metadatos técnicos solo pasa `_row_hash`, que basta para rastrear cada fila hasta TRANSFORMED; los demás (`_batch_id`, `_extracted_at`, `_source_table`, `_transformed_at`, `_transform_version`) se quedan en el Parquet.

**df1 — `transformed_vg_ventas_ratings`** (grano: ventas de un juego en una plataforma)

| Columnas de la tabla plana | Se vuelven… |
|---|---|
| `name`, `genre`, `categoria` | `dim_juego_df1` |
| `platform`, `fabricante` | `dim_plataforma_df1` |
| `publisher` | `dim_editorial_df1` |
| `developer` | `dim_desarrollador_df1` (nulo → `'Desconocido'`) |
| `rating` (ESRB) | `dim_clasificacion_df1` (nulo → `'Sin clasificación'`) |
| `year_of_release`, `decada` | `dim_tiempo_df1` (año nulo → `anio = 0`, *desconocido*) |
| `na_sales` … `global_sales`, `critic_score`, `critic_count`, `user_score`, `user_count` | `fact_ventas_df1` (medidas) |
| `mercado_principal`, `ventas_consistentes`, `_row_hash` | `fact_ventas_df1` (atributos degenerados) |

**df2 — `transformed_vg_global_codificado`** (grano: un registro de ventas; **no trae nombre de juego ni año**)

| Columnas de la tabla plana | Se vuelven… |
|---|---|
| `platform`, `plataforma`, `fabricante` | `dim_plataforma_df2` (el código ya existe y se usa como llave) |
| `genre`, `genero`, `categoria` | `dim_genero_df2` |
| `publisher` | `dim_editorial_df2` (solo viene el código) |
| `rating`, `clasificacion` | `dim_clasificacion_df2` |
| `critic_score_class` | `dim_clase_critica_df2` (solo viene el código) |
| `na_sales` … `global_sales`, `ventas_fuera_na` | `fact_ventas_df2` (medidas) |
| `mercado_principal`, `ventas_consistentes`, `_row_hash` | `fact_ventas_df2` (atributos degenerados) |

**df3 — `transformed_raw_vg_engagement`** (grano: un juego listado en la plataforma de reseñas; **no trae ventas**)

| Columnas de la tabla plana | Se vuelven… |
|---|---|
| `title`, `team` | `dim_juego_df3` |
| `genero_principal`, `categoria` | `dim_genero_df3` |
| `anio`, `decada` | `dim_tiempo_df3` |
| `rating`, `times_listed`, `number_of_reviews`, `plays`, `playing`, `backlogs`, `wishlist` | `fact_engagement_df3` (medidas) |
| `release_date`, `genres`, `playing_consistente`, `_row_hash` | `fact_engagement_df3` (atributos degenerados) |

---
## PASO 3 — Construir el esquema estrella (3.1 físico + 3.2)

```
          dim_clasificacion_df1        dim_tiempo_df1
                         \               /
   dim_juego_df1 ──────── fact_ventas_df1 ──────── dim_plataforma_df1
                         /               \
          dim_desarrollador_df1        dim_editorial_df1
```

| Tipo de tabla | Qué guarda | Filas |
|---|---|---|
| **Hechos** (`fact_`) | Lo que se **mide**: números que se suman | Muchas |
| **Dimensiones** (`dim_`) | El **contexto** para filtrar y agrupar: qué, quién, cuándo | Pocas |

**Grano (df1):** una fila de `fact_ventas_df1` = las ventas de un juego en una plataforma (igual que la tabla plana).

**Diseño físico** — lo escribimos en cada `CREATE TABLE`:

| Elemento | Para qué |
|---|---|
| **Tipo de dato** (`INTEGER`, `VARCHAR`, `DOUBLE`) | Cada columna se guarda con su tipo definitivo (`anio` → `INTEGER`) |
| `PRIMARY KEY` | Identifica cada fila; no se repite |
| `ROW_NUMBER()` | Genera la llave numérica (sustituta) de cada dimensión |
| `REFERENCES` | Llave foránea: el hecho apunta a una dimensión que sí existe |

In [ ]:
# ── Empezar de cero ────────────────────────────────────────────────────────
# Primero se borran los hechos, porque apuntan a las dimensiones.

for tabla in ['fact_ventas_df1', 'fact_ventas_df2', 'fact_engagement_df3',
              'dim_juego_df1', 'dim_plataforma_df1', 'dim_editorial_df1', 'dim_tiempo_df1',
              'dim_clasificacion_df1', 'dim_desarrollador_df1',
              'dim_plataforma_df2', 'dim_genero_df2', 'dim_editorial_df2', 'dim_clasificacion_df2',
              'dim_clase_critica_df2',
              'dim_juego_df3', 'dim_genero_df3', 'dim_tiempo_df3']:
    con.execute(f'DROP TABLE IF EXISTS {tabla}')

print()
print('Listo para construir los modelos')
print()

### 3.1 `dim_plataforma_df1` (df1)

Cada dimensión se hace en dos pasos: **crear** la tabla (tipos y llaves) e **insertar** los valores distintos de la tabla plana.

Cada plataforma pertenece a un solo fabricante, así que `fabricante` va en la misma dimensión.

In [ ]:
# ── dim_plataforma_df1 ─────────────────────────────────────────────────────

con.execute('''

    CREATE TABLE dim_plataforma_df1 (
        plataforma_id   INTEGER PRIMARY KEY,
        plataforma      VARCHAR NOT NULL UNIQUE,
        fabricante      VARCHAR NOT NULL
    )
''')




con.execute('''

    INSERT INTO dim_plataforma_df1
    SELECT ROW_NUMBER() OVER (ORDER BY platform) AS plataforma_id,
           platform                              AS plataforma,
           fabricante
    FROM (SELECT DISTINCT platform, fabricante FROM base_df1)
''')

con.sql('SELECT * FROM dim_plataforma_df1 LIMIT 5').df()

### 3.2 `dim_editorial_df1` (df1)

In [ ]:
# ── dim_editorial_df1 ──────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_editorial_df1 (
        editorial_id    INTEGER PRIMARY KEY,
        editorial       VARCHAR NOT NULL UNIQUE
    )
''')

con.execute('''
    INSERT INTO dim_editorial_df1
    SELECT ROW_NUMBER() OVER (ORDER BY publisher) AS editorial_id,
           publisher                              AS editorial
    FROM (SELECT DISTINCT publisher FROM base_df1)
''')

con.sql('SELECT * FROM dim_editorial_df1 LIMIT 5').df()

### 3.3 `dim_tiempo_df1` (df1)

La llave es el propio año (`year_of_release`, ya es entero). Agregamos la década para poder agrupar.

**269 juegos no tienen año.** Una llave primaria no puede ser nula, así que se usa el **miembro desconocido** `anio = 0` (`COALESCE`). Así ninguna venta se pierde.

In [ ]:
# ── dim_tiempo_df1 ─────────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_tiempo_df1 (
        anio      INTEGER PRIMARY KEY,
        decada    INTEGER NOT NULL
    )
''')

con.execute('''
    INSERT INTO dim_tiempo_df1
    SELECT DISTINCT CAST(COALESCE(year_of_release, 0) AS INTEGER)            AS anio,
                    CAST(COALESCE(year_of_release, 0) AS INTEGER) // 10 * 10 AS decada
    FROM base_df1
''')

con.sql('SELECT * FROM dim_tiempo_df1 ORDER BY anio LIMIT 5').df()

### 3.4 `dim_juego_df1` (df1)

Algunos juegos aparecen con dos géneros, así que el juego se identifica por **nombre + género** (`UNIQUE (nombre, genero)`). La `categoria` depende solo del género, así que también va aquí.

In [ ]:
# ── dim_juego_df1 ──────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_juego_df1 (
        juego_id    INTEGER PRIMARY KEY,
        nombre      VARCHAR NOT NULL,
        genero      VARCHAR NOT NULL,
        categoria   VARCHAR NOT NULL,
        UNIQUE (nombre, genero)
    )
''')

con.execute('''
    INSERT INTO dim_juego_df1
    SELECT ROW_NUMBER() OVER (ORDER BY name, genre)   AS juego_id,
           name                                       AS nombre,
           genre                                      AS genero,
           categoria
    FROM (SELECT DISTINCT name, genre, categoria FROM base_df1)
''')

con.sql('SELECT * FROM dim_juego_df1 LIMIT 5').df()

### 3.5 `dim_clasificacion_df1` (df1)

Clasificación ESRB (`E`, `T`, `M`…). **6,764 juegos no la tienen**: igual que con el año, se usa un **miembro desconocido**, `'Sin clasificación'`, para que ninguna venta se pierda.

In [ ]:
# ── dim_clasificacion_df1 ──────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_clasificacion_df1 (
        clasificacion_id   INTEGER PRIMARY KEY,
        clasificacion      VARCHAR NOT NULL UNIQUE
    )
''')

con.execute('''
    INSERT INTO dim_clasificacion_df1
    SELECT ROW_NUMBER() OVER (ORDER BY clasificacion) AS clasificacion_id,
           clasificacion
    FROM (SELECT DISTINCT COALESCE(rating, 'Sin clasificación') AS clasificacion FROM base_df1)
''')

con.sql('SELECT * FROM dim_clasificacion_df1').df()

### 3.6 `dim_desarrollador_df1` (df1)

El estudio que desarrolló el juego (puede ser distinto de la editorial). **6,618 juegos no lo tienen** → miembro desconocido `'Desconocido'`.

In [ ]:
# ── dim_desarrollador_df1 ──────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_desarrollador_df1 (
        desarrollador_id   INTEGER PRIMARY KEY,
        desarrollador      VARCHAR NOT NULL UNIQUE
    )
''')

con.execute('''
    INSERT INTO dim_desarrollador_df1
    SELECT ROW_NUMBER() OVER (ORDER BY desarrollador) AS desarrollador_id,
           desarrollador
    FROM (SELECT DISTINCT COALESCE(developer, 'Desconocido') AS desarrollador FROM base_df1)
''')

con.sql('SELECT * FROM dim_desarrollador_df1 LIMIT 5').df()

### 3.7 `fact_ventas_df1` (df1) — el centro de la estrella

Guarda **llaves, medidas y atributos degenerados**. Cada llave se obtiene uniendo (`JOIN`) la tabla plana con su dimensión por el texto original. Como el dataset no trae `Rank`, `venta_id` se numera por ventas globales.

Las calificaciones (`critic_score`, `user_score`) y sus conteos son **medidas**: se promedian o se suman por cualquier dimensión. Se dejan nulas cuando el juego no fue reseñado (decisión de la Unidad 2).

In [ ]:
# ── fact_ventas_df1 ────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE fact_ventas_df1 (

        venta_id            INTEGER PRIMARY KEY,

        juego_id            INTEGER NOT NULL REFERENCES dim_juego_df1 (juego_id),
        plataforma_id       INTEGER NOT NULL REFERENCES dim_plataforma_df1 (plataforma_id),
        editorial_id        INTEGER NOT NULL REFERENCES dim_editorial_df1 (editorial_id),
        desarrollador_id    INTEGER NOT NULL REFERENCES dim_desarrollador_df1 (desarrollador_id),
        clasificacion_id    INTEGER NOT NULL REFERENCES dim_clasificacion_df1 (clasificacion_id),
        anio                INTEGER NOT NULL REFERENCES dim_tiempo_df1 (anio),

        ventas_na           DOUBLE,
        ventas_eu           DOUBLE,
        ventas_jp           DOUBLE,
        ventas_otras        DOUBLE,
        ventas_global       DOUBLE,
        puntaje_critica     INTEGER,
        num_criticas        INTEGER,
        puntaje_usuarios    DOUBLE,
        num_usuarios        INTEGER,

        mercado_principal   VARCHAR,
        ventas_consistentes BOOLEAN,
        _row_hash           VARCHAR
    )
''')



con.execute('''

    INSERT INTO fact_ventas_df1

    SELECT ROW_NUMBER() OVER (ORDER BY b.global_sales DESC, b._row_hash)   AS venta_id,

           j.juego_id,
           p.plataforma_id,
           e.editorial_id,
           d.desarrollador_id,
           c.clasificacion_id,
           CAST(COALESCE(b.year_of_release, 0) AS INTEGER)                 AS anio,

           b.na_sales, b.eu_sales, b.jp_sales, b.other_sales, b.global_sales,
           b.critic_score, b.critic_count, b.user_score, b.user_count,

           b.mercado_principal,
           b.ventas_consistentes,
           b._row_hash


    FROM base_df1 AS b


    JOIN dim_juego_df1         AS j ON j.nombre = b.name AND j.genero = b.genre
    JOIN dim_plataforma_df1    AS p ON p.plataforma = b.platform
    JOIN dim_editorial_df1     AS e ON e.editorial  = b.publisher
    JOIN dim_desarrollador_df1 AS d ON d.desarrollador = COALESCE(b.developer, 'Desconocido')
    JOIN dim_clasificacion_df1 AS c ON c.clasificacion = COALESCE(b.rating, 'Sin clasificación')


''')

con.sql('SELECT * FROM fact_ventas_df1 LIMIT 5').df()

---
### 3.8 Modelo df2 — `transformed_vg_global_codificado`

Este dataset **ya trae códigos numéricos** (`platform`, `genre`, `publisher`, `rating`, `critic_score_class`) junto a su texto, con relación 1 a 1. Por eso el código se usa directo como **llave primaria** (no hace falta `ROW_NUMBER()`) y el hecho no necesita `JOIN` para obtener sus llaves.

`publisher` y `critic_score_class` no tienen texto (en la Unidad 2 no se pudieron decodificar), así que su dimensión solo guarda el código: sirven para **agrupar**, aunque todavía no digan el nombre.

No trae nombre de juego ni año → este modelo no tiene `dim_juego` ni `dim_tiempo`.

```
          dim_clasificacion_df2       dim_clase_critica_df2
                         \               /
  dim_genero_df2 ──────── fact_ventas_df2 ──────── dim_plataforma_df2
                                 │
                         dim_editorial_df2
```

In [ ]:
# ── Dimensiones df2 ────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_plataforma_df2 (
        plataforma_id   INTEGER PRIMARY KEY,
        plataforma      VARCHAR NOT NULL UNIQUE,
        fabricante      VARCHAR NOT NULL
    )
''')
con.execute('''
    INSERT INTO dim_plataforma_df2
    SELECT DISTINCT platform, plataforma, fabricante FROM base_df2
''')


con.execute('''
    CREATE TABLE dim_genero_df2 (
        genero_id       INTEGER PRIMARY KEY,
        genero          VARCHAR NOT NULL UNIQUE,
        categoria       VARCHAR NOT NULL
    )
''')
con.execute('''
    INSERT INTO dim_genero_df2
    SELECT DISTINCT genre, genero, categoria FROM base_df2
''')


con.execute('''
    CREATE TABLE dim_editorial_df2 (
        editorial_id    INTEGER PRIMARY KEY
    )
''')
con.execute('''
    INSERT INTO dim_editorial_df2
    SELECT DISTINCT publisher FROM base_df2
''')


con.execute('''
    CREATE TABLE dim_clasificacion_df2 (
        clasificacion_id INTEGER PRIMARY KEY,
        clasificacion    VARCHAR NOT NULL UNIQUE
    )
''')
con.execute('''
    INSERT INTO dim_clasificacion_df2
    SELECT DISTINCT rating, clasificacion FROM base_df2
''')

con.execute('''
    CREATE TABLE dim_clase_critica_df2 (
        clase_critica_id INTEGER PRIMARY KEY
    )
''')
con.execute('''
    INSERT INTO dim_clase_critica_df2
    SELECT DISTINCT critic_score_class FROM base_df2
''')

con.sql('SELECT * FROM dim_plataforma_df2 ORDER BY plataforma_id LIMIT 5').df()

In [ ]:
# ── fact_ventas_df2 ────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE fact_ventas_df2 (

        venta_id            INTEGER PRIMARY KEY,

        plataforma_id       INTEGER NOT NULL REFERENCES dim_plataforma_df2 (plataforma_id),
        genero_id           INTEGER NOT NULL REFERENCES dim_genero_df2 (genero_id),
        editorial_id        INTEGER NOT NULL REFERENCES dim_editorial_df2 (editorial_id),
        clasificacion_id    INTEGER NOT NULL REFERENCES dim_clasificacion_df2 (clasificacion_id),
        clase_critica_id    INTEGER NOT NULL REFERENCES dim_clase_critica_df2 (clase_critica_id),

        ventas_na           DOUBLE,
        ventas_eu           DOUBLE,
        ventas_jp           DOUBLE,
        ventas_otras        DOUBLE,
        ventas_global       DOUBLE,
        ventas_fuera_na     DOUBLE,

        mercado_principal   VARCHAR,
        ventas_consistentes BOOLEAN,
        _row_hash           VARCHAR
    )
''')

con.execute('''
    INSERT INTO fact_ventas_df2
    SELECT ROW_NUMBER() OVER (ORDER BY global_sales DESC, _row_hash) AS venta_id,
           platform, genre, publisher, rating, critic_score_class,
           na_sales, eu_sales, jp_sales, other_sales, global_sales, ventas_fuera_na,
           mercado_principal, ventas_consistentes,
           _row_hash
    FROM base_df2
''')

con.sql('SELECT * FROM fact_ventas_df2 LIMIT 5').df()

---
### 3.9 Modelo df3 — `transformed_raw_vg_engagement`

Aquí no hay ventas: las medidas son de **engagement** (reseñas, jugadores, *wishlist*…), así que el hecho se llama `fact_engagement_df3`.

El título **no es único** (por ejemplo, hay dos *Tetris* de Nintendo), así que el juego se identifica por **título + equipo** (`UNIQUE (titulo, equipo)`). El género y el año van en su propia dimensión.

`genres` (la lista completa de géneros) **no puede ir en `dim_juego_df3`**: un mismo título + equipo aparece con dos listas distintas y rompería la llave. Por eso queda en el hecho como atributo degenerado, junto con `fecha_lanzamiento` y la bandera `playing_consistente`.

```
                  dim_tiempo_df3
                        │
  dim_juego_df3 ── fact_engagement_df3 ── dim_genero_df3
```

In [ ]:
# ── Dimensiones df3 ────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_juego_df3 (
        juego_id    INTEGER PRIMARY KEY,
        titulo      VARCHAR NOT NULL,
        equipo      VARCHAR NOT NULL,
        UNIQUE (titulo, equipo)
    )
''')
con.execute('''
    INSERT INTO dim_juego_df3
    SELECT ROW_NUMBER() OVER (ORDER BY title, team) AS juego_id,
           title                                    AS titulo,
           team                                     AS equipo
    FROM (SELECT DISTINCT title, team FROM base_df3)
''')


con.execute('''
    CREATE TABLE dim_genero_df3 (
        genero_id   INTEGER PRIMARY KEY,
        genero      VARCHAR NOT NULL UNIQUE,
        categoria   VARCHAR NOT NULL
    )
''')
con.execute('''
    INSERT INTO dim_genero_df3
    SELECT ROW_NUMBER() OVER (ORDER BY genero_principal) AS genero_id,
           genero_principal                              AS genero,
           categoria
    FROM (SELECT DISTINCT genero_principal, categoria FROM base_df3)
''')


con.execute('''
    CREATE TABLE dim_tiempo_df3 (
        anio      INTEGER PRIMARY KEY,
        decada    INTEGER NOT NULL
    )
''')
con.execute('''
    INSERT INTO dim_tiempo_df3
    SELECT DISTINCT anio, decada FROM base_df3
''')

con.sql('SELECT * FROM dim_juego_df3 LIMIT 5').df()

In [ ]:
# ── fact_engagement_df3 ────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE fact_engagement_df3 (

        registro_id         INTEGER PRIMARY KEY,

        juego_id            INTEGER NOT NULL REFERENCES dim_juego_df3 (juego_id),
        genero_id           INTEGER NOT NULL REFERENCES dim_genero_df3 (genero_id),
        anio                INTEGER NOT NULL REFERENCES dim_tiempo_df3 (anio),

        rating              DOUBLE,
        veces_listado       BIGINT,
        num_resenas         BIGINT,
        jugadores           BIGINT,
        jugando             BIGINT,
        pendientes          BIGINT,
        wishlist            BIGINT,

        fecha_lanzamiento   DATE,
        generos             VARCHAR,
        jugando_consistente BOOLEAN,
        _row_hash           VARCHAR
    )
''')

con.execute('''
    INSERT INTO fact_engagement_df3
    SELECT ROW_NUMBER() OVER (ORDER BY b.plays DESC, b._row_hash) AS registro_id,
           j.juego_id,
           g.genero_id,
           b.anio,
           b.rating, b.times_listed, b.number_of_reviews,
           b.plays, b.playing, b.backlogs, b.wishlist,
           CAST(b.release_date AS DATE)                           AS fecha_lanzamiento,
           b.genres                                               AS generos,
           b.playing_consistente                                  AS jugando_consistente,
           b._row_hash
    FROM base_df3 AS b
    JOIN dim_juego_df3  AS j ON j.titulo = b.title AND j.equipo = b.team
    JOIN dim_genero_df3 AS g ON g.genero = b.genero_principal
''')

con.sql('SELECT * FROM fact_engagement_df3 LIMIT 5').df()

In [ ]:
# ── Resultado: el esquema estrella en serving.db ───────────────────────────

con.sql('SHOW TABLES').df()

In [ ]:
# ── Ver una tabla del esquema estrella ─────────────────────────────────────

TABLA = 'dim_juego_df1'                                           # ← cambia: cualquier tabla de SHOW TABLES (fact_ventas_df2, dim_juego_df3, ...)

print()
print(f'Filas: {con.sql(f"SELECT COUNT(*) FROM {TABLA}").fetchall()[0][0]:,}')
print()

con.sql(f'SELECT * FROM {TABLA}').df()

---
## PASO 4 — Validar el modelo

Al repartir los datos en tablas **no se debe perder ni duplicar nada**: mismas filas, mismas ventas y ninguna venta sin dimensión (*huérfana*).

In [ ]:
# ── Filas, ventas y calificaciones: antes contra después (df1) ─────────────

con.sql('''

    SELECT (SELECT COUNT(*) FROM base_df1)                               AS filas_tabla_plana,
           (SELECT COUNT(*) FROM fact_ventas_df1)                        AS filas_fact_ventas_df1,

           (SELECT ROUND(SUM(global_sales), 2) FROM base_df1)            AS ventas_tabla_plana,
           (SELECT ROUND(SUM(ventas_global), 2) FROM fact_ventas_df1)    AS ventas_fact_ventas_df1,

           (SELECT ROUND(AVG(critic_score), 2) FROM base_df1)            AS critica_prom_tabla_plana,
           (SELECT ROUND(AVG(puntaje_critica), 2) FROM fact_ventas_df1)  AS critica_prom_fact_ventas_df1
''').df()

In [ ]:
# ── Filas y ventas: antes contra después (df2) ─────────────────────────────

con.sql('''
    SELECT (SELECT COUNT(*) FROM base_df2)                            AS filas_tabla_plana,
           (SELECT COUNT(*) FROM fact_ventas_df2)                     AS filas_fact_ventas_df2,
           (SELECT ROUND(SUM(global_sales), 2) FROM base_df2)         AS ventas_tabla_plana,
           (SELECT ROUND(SUM(ventas_global), 2) FROM fact_ventas_df2) AS ventas_fact_ventas_df2
''').df()

In [ ]:
# ── Filas y jugadores: antes contra después (df3) ──────────────────────────

con.sql('''
    SELECT (SELECT COUNT(*) FROM base_df3)                    AS filas_tabla_plana,
           (SELECT COUNT(*) FROM fact_engagement_df3)         AS filas_fact_engagement_df3,
           (SELECT SUM(plays) FROM base_df3)                  AS jugadores_tabla_plana,
           (SELECT SUM(jugadores) FROM fact_engagement_df3)   AS jugadores_fact_engagement_df3
''').df()

In [ ]:
# ── Registros huérfanos (todos deben ser 0) ────────────────────────────────

con.sql('''

    SELECT
      (SELECT COUNT(*)
       FROM fact_ventas_df1 AS f
       LEFT JOIN dim_juego_df1         AS j USING (juego_id)
       LEFT JOIN dim_plataforma_df1    AS p USING (plataforma_id)
       LEFT JOIN dim_editorial_df1     AS e USING (editorial_id)
       LEFT JOIN dim_desarrollador_df1 AS d USING (desarrollador_id)
       LEFT JOIN dim_clasificacion_df1 AS c USING (clasificacion_id)
       LEFT JOIN dim_tiempo_df1        AS t USING (anio)
       WHERE j.juego_id IS NULL OR p.plataforma_id IS NULL
          OR e.editorial_id IS NULL OR d.desarrollador_id IS NULL
          OR c.clasificacion_id IS NULL OR t.anio IS NULL)          AS huerfanas_df1,

      (SELECT COUNT(*)
       FROM fact_ventas_df2 AS f
       LEFT JOIN dim_plataforma_df2    AS p USING (plataforma_id)
       LEFT JOIN dim_genero_df2        AS g USING (genero_id)
       LEFT JOIN dim_editorial_df2     AS e USING (editorial_id)
       LEFT JOIN dim_clasificacion_df2 AS c USING (clasificacion_id)
       LEFT JOIN dim_clase_critica_df2 AS k USING (clase_critica_id)
       WHERE p.plataforma_id IS NULL OR g.genero_id IS NULL
          OR e.editorial_id IS NULL OR c.clasificacion_id IS NULL
          OR k.clase_critica_id IS NULL)                            AS huerfanas_df2,

      (SELECT COUNT(*)
       FROM fact_engagement_df3 AS f
       LEFT JOIN dim_juego_df3  AS j USING (juego_id)
       LEFT JOIN dim_genero_df3 AS g USING (genero_id)
       LEFT JOIN dim_tiempo_df3 AS t USING (anio)
       WHERE j.juego_id IS NULL OR g.genero_id IS NULL
          OR t.anio IS NULL)                                        AS huerfanas_df3

''').df()

---
## PASO 5 — Consultar el modelo

Toda consulta sobre una estrella tiene la misma forma: **los hechos al centro, unidos a las dimensiones que pide la pregunta**.

In [ ]:
# ── df1: ¿Qué géneros venden más en cada década? ───────────────────────────

con.sql('''

    SELECT t.decada,
           j.genero,
           ROUND(SUM(f.ventas_global), 2) AS ventas_millones

    FROM fact_ventas_df1 AS f

    JOIN dim_juego_df1   AS j USING (juego_id)
    JOIN dim_tiempo_df1  AS t USING (anio)

    GROUP BY t.decada, j.genero
    ORDER BY t.decada, ventas_millones DESC

''').df()

In [ ]:
# ── df1: ¿Cómo califica la crítica a cada clasificación ESRB? ──────────────

con.sql('''

    SELECT c.clasificacion,
           COUNT(f.puntaje_critica)            AS juegos_reseñados,
           ROUND(AVG(f.puntaje_critica), 1)    AS critica_promedio,
           ROUND(AVG(f.puntaje_usuarios), 2)   AS usuarios_promedio,
           ROUND(SUM(f.ventas_global), 2)      AS ventas_millones

    FROM fact_ventas_df1 AS f

    JOIN dim_clasificacion_df1 AS c USING (clasificacion_id)

    GROUP BY c.clasificacion
    ORDER BY ventas_millones DESC

''').df()

In [ ]:
# ── df2: ¿Qué fabricante vende más en cada categoría de juego? ─────────────

con.sql('''

    SELECT p.fabricante,
           g.categoria,
           ROUND(SUM(f.ventas_global), 2) AS ventas_millones

    FROM fact_ventas_df2 AS f

    JOIN dim_plataforma_df2 AS p USING (plataforma_id)
    JOIN dim_genero_df2     AS g USING (genero_id)

    GROUP BY p.fabricante, g.categoria
    ORDER BY p.fabricante, ventas_millones DESC

''').df()

In [ ]:
# ── df3: ¿Qué géneros tienen más jugadores en cada década? ─────────────────

con.sql('''

    SELECT t.decada,
           g.genero,
           SUM(f.jugadores)          AS jugadores,
           ROUND(AVG(f.rating), 2)   AS rating_promedio

    FROM fact_engagement_df3 AS f

    JOIN dim_genero_df3 AS g USING (genero_id)
    JOIN dim_tiempo_df3 AS t USING (anio)

    GROUP BY t.decada, g.genero
    ORDER BY t.decada, jugadores DESC

''').df()

In [ ]:
# ── Cerrar la conexión ─────────────────────────────────────────────────────

con.close()

print()
print(f'Modelo guardado en: {SERVING_DB}')
print()

In [ ]:
ver_serving(SERVING_DB)

---
### Resumen

| Paso | Tema | Resultado |
|---|---|---|
| 1 | — | Vistas `base_df1`, `base_df2`, `base_df3` sobre los Parquet |
| 2 | 3.1 Lógico | Qué columnas son dimensiones y cuáles hechos, por dataset |
| 3 | 3.1 Físico + 3.2 | **Tres esquemas estrella** en `serving.db` |
| 4 | — | Mismas filas, mismas medidas, 0 huérfanas |
| 5 | — | Una consulta hechos + dimensiones por modelo |

| Dataset | Grano | Hecho | Dimensiones |
|---|---|---|---|
| df1 `vg_ventas_ratings` | Ventas de un juego en una plataforma | `fact_ventas_df1` | `dim_juego_df1`, `dim_plataforma_df1`, `dim_editorial_df1`, `dim_desarrollador_df1`, `dim_clasificacion_df1`, `dim_tiempo_df1` |
| df2 `vg_global_codificado` | Un registro de ventas codificado | `fact_ventas_df2` | `dim_plataforma_df2`, `dim_genero_df2`, `dim_editorial_df2`, `dim_clasificacion_df2`, `dim_clase_critica_df2` |
| df3 `raw_vg_engagement` | Un juego listado en la plataforma de reseñas | `fact_engagement_df3` | `dim_juego_df3`, `dim_genero_df3`, `dim_tiempo_df3` |